# SAPO Evaluation Test
Qwen2.5-7B-Instruct로 sapo train.jsonl 25개 전체 평가

In [ ]:
!pip install unsloth transformers torch

In [ ]:
import json
import re
import sys
from pathlib import Path

sys.path.insert(0, ".")
from test import (
    check_regex_present,
    check_order,
    count_pattern,
    extract_between,
    evaluate_category,
    evaluate_segments,
)

In [ ]:
from unsloth import FastLanguageModel

MODEL_PATH = "unsloth/Qwen2.5-7B-Instruct"
# MODEL_PATH = "./path/to/finetuned"  # local fine-tuned model

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

In [ ]:
records = []
with open("dataset/sapo/train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"Loaded {len(records)} items")
for r in records:
    cl_count = len(r["checklist"])
    print(f"  [{r['category']}] #{r['index']} — checklist: {cl_count} items")

In [ ]:
def build_dynamic_checklist(checklist_dict: dict) -> list:
    """jsonl checklist dict -> test.py 호환 check_fn 리스트 변환"""

    simple_patterns = []
    for k in sorted(checklist_dict.keys(), key=lambda x: int(x)):
        p = checklist_dict[k]["pattern"]
        if not _is_complex(p):
            simple_patterns.append(p.split("|")[0])

    items = []
    for k in sorted(checklist_dict.keys(), key=lambda x: int(x)):
        entry = checklist_dict[k]
        pattern = entry["pattern"]
        fn = _pattern_to_fn(pattern, simple_patterns)
        items.append({
            "id": int(k),
            "description": entry["description"],
            "check_fn": fn,
        })
    return items


def _is_complex(pattern: str) -> bool:
    if pattern in ("order_check", "markdown_headings", "list_of_dicts_check"):
        return True
    if re.match(r"\d+_\w+", pattern) and "|" not in pattern:
        return True
    if pattern.startswith("budget_sum") or pattern.startswith("day_order"):
        return True
    return False


def _pattern_to_fn(pattern: str, ordered_simple_patterns: list):

    # --- order_check: 이전 simple 패턴들의 등장 순서 확인 ---
    if pattern in ("order_check", "day_order_check"):
        return lambda text, ops=ordered_simple_patterns: check_order(text, ops)

    # --- budget_sum_lte_N ---
    m = re.match(r"budget_sum_lte_(\d+)", pattern)
    if m:
        mx = int(m.group(1))
        def _budget_check(text, mx=mx):
            amounts = re.findall(r"\$\s*([\d,]+(?:\.\d{1,2})?)", text)
            if not amounts:
                return False
            return sum(float(a.replace(",", "")) for a in amounts) <= mx
        return _budget_check

    # --- N_something: 섹션 내 항목 N개 이상 ---
    m = re.match(r"(\d+)_(\w+)", pattern)
    if m and "|" not in pattern:
        n = int(m.group(1))
        kind = m.group(2)
        item_pattern = r"[-*•]\s+\w|\d+[\.)\]]\s+\w"
        if "time" in kind:
            item_pattern = r"\d{1,2}:\d{2}\s*(?:AM|PM|am|pm)?"
        elif "email" in kind:
            item_pattern = r"[\w.+-]+@[\w-]+\.[\w.-]+"
        elif "docstring" in kind:
            item_pattern = r'"""[\s\S]*?"""|\x27\x27\x27[\s\S]*?\x27\x27\x27'
        elif "test_function" in kind:
            item_pattern = r"def\s+test_"
        elif "assert" in kind:
            item_pattern = r"\bassert\s+"
        return lambda text, n=n, p=item_pattern: count_pattern(text, p) >= n

    # --- markdown_headings ---
    if pattern == "markdown_headings":
        return lambda text: count_pattern(text, r"^#{1,6}\s+") >= 3

    # --- list_of_dicts_check ---
    if pattern == "list_of_dicts_check":
        return lambda text: check_regex_present(text, r"\[.*\{.*\}.*\]")

    # --- 110_and_119 (emergency numbers) ---
    if pattern == "110_and_119":
        return lambda text: check_regex_present(text, r"110") and check_regex_present(text, r"119")

    # --- default: simple regex ---
    return lambda text, p=pattern: check_regex_present(text, p)

In [ ]:
def generate_response(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    outputs = model.generate(
        inputs,
        max_new_tokens=4096,
        temperature=0.7,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

In [ ]:
results = []

for i, record in enumerate(records):
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(records)}] {record['category']} #{record['index']}")
    print(f"{'='*60}")

    response = generate_response(record["question"])
    checklist = build_dynamic_checklist(record["checklist"])

    score, details = evaluate_category(response, checklist)
    seg_result = evaluate_segments(response, checklist)

    results.append({
        "category": record["category"],
        "index": record["index"],
        "score": score,
        "details": details,
        "seg_result": seg_result,
        "response": response,
    })

    print(f"Score: {score:.2%} ({sum(d['passed'] for d in details)}/{len(details)})")
    for d in details:
        status = "\u2705" if d["passed"] else "\u274c"
        print(f"  {status} [{d['id']}] {d['description']}")

    if seg_result["segments"]:
        breathe_count = sum(1 for s in seg_result["segments"] if s["type"] == "breathe")
        print(f"  Segments: {len(seg_result['segments'])} (breathe: {breathe_count})")

In [ ]:
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}\n")

by_category = {}
for r in results:
    by_category.setdefault(r["category"], []).append(r["score"])
    print(f"  [{r['category']}] #{r['index']:>2d}  →  {r['score']:.2%}")

print(f"\n{'-'*60}")
for cat, scores in by_category.items():
    avg = sum(scores) / len(scores)
    print(f"  {cat}: {avg:.2%} (n={len(scores)})")

total_avg = sum(r["score"] for r in results) / len(results)
print(f"\n  Overall Average: {total_avg:.2%} ({len(results)} items)")